## RAG 최적화 경진대회

여러분이 만든 챗봇이 매우 좋은 반응을 얻은 덕분에,  
**한국전력공사 관련 질문에도 답변할 수 있도록 챗봇을 확장해줄 수 있냐**는 새로운 요청이 들어왔습니다.

이번 미션에서는 여러분의 챗봇이  
SK하이닉스뿐 아니라 한국전력공사의 지속가능경영보고서까지 반영하여, 
사용자의 질문에 정확하게 답할 수 있도록 개선해야 합니다.  

필요한 데이터는 모두 확보되어 있으며,  
질문이 SK하이닉스 관련인지 한전 관련인지 적절히 구분한 뒤,  
각각의 문서에 기반하여 정확한 답변을 제공해야 합니다.  

💬 질문: 보고서 발행일은 언제인가요? # p.2 | 2025년 6월 19일

💬 질문: 보고 기간은 언제부터 언제까지인가요? # p.2 | 2024년 1월 1일~2024년 12월 31일

💬 질문: 이해관계자 구분은 몇 개 그룹인가요? # p.11 | 7개 그룹

💬 질문: 고객 소통 채널 중 하나는 무엇인가요? # p.11 | 홈페이지

💬 질문: 장애인/저소득층 일자리 창출 개수는? # p.16 | 1000개

💬 질문: Global ICT 인재 육성 프로그램 참여 누적 인원은? # p.16 | 4만 7679명

💬 질문: 온실가스 감축 3개년 로드맵 발표 연도는? # p.21 | 2022년

💬 질문: SCC 결성 연도는? # p.21 | 2022년

💬 질문: SK하이닉스의 재활용 소재 사용 목표 연도는? # p.27 | 2024년

💬 질문: 2030년까지 재활용 소재 사용률 목표는? # p.27 | 30%

💬 질문: 관련 중대 이슈는 무엇인가요? # p.32 | 공급망 관리 등

💬 질문: 우리 회원 수는 몇 명인가요? # p.32 | 33

💬 질문: 출산 축하금은 얼마인가요? # p.37 | 첫째·둘째 100만 원, 셋째 500만 원

💬 질문: 특별육아휴직 기간은 얼마인가요? # p.37 | 1년

💬 질문: 2024년 정기 작업 위험성평가는 몇 건을 평가했나요? # p.42 | 3만 9000건

💬 질문: 개선 전 평균 위험도는 몇 점이었나요? # p.42 | 5.3점

💬 질문: 2024년 하반기 책임광물 조사 결과에서 주석 협력사 수는? # p.47 | 56

💬 질문: 2024년 개선 컨설팅 지원 협력사 수는? # p.47 | 16

💬 질문: 행복나눔기금의 누적 조성액은 얼마인가요? # p.52 | 약 346억 원

💬 질문: 2024년 행복도시락 사업에서 몇 명의 아동에게 도시락을 제공했나요? # p.52 | 810명

💬 질문: SK하이닉스의 On-Site 협업 자문 프로그램은 무엇을 제공하나? # p.56 | 산업체 현안 이해

💬 질문: 장비 가동을 위한 조치 시간은 몇 분인가? # p.56 | 19분

💬 질문: 어떤 회사가 2025년 World's Most Ethical Companies에 선정됐나요? # p.62 | SK하이닉스

💬 질문: World's Most Ethical Companies 선정에 사용된 질문 수는 몇 개인가요? # p.62 | 240여 개

💬 질문: 이사 보수는 어떤 요소로 구성되나요? # p.67 | 기본급, 성과급, 기타 소득

💬 질문: 사내이사 보수는 누구에게 위임되나요? # p.67 | 인사·보상위원회

💬 질문: 2023년 국내 취수량은 얼마인가요? # p.74 | 74,146 천 m³

💬 질문: 2024년 해외 소비량은 얼마인가요? # p.74 | 3,054 천 m³

💬 질문: 2021년 Scope 1 총배출량은? # p.83 | 2,628,921 tCO2eq

💬 질문: 2023년 재생에너지 비율은? # p.83 | 19%

### 💡미션
완성된 챗봇은 SK하이닉스 지속가능경영보고서 및 한국전력공사 지속경영보고서를 기반으로 질문에 답변해야 합니다
- 문서에 정보가 없는 경우는 "정보 없음"으로 답변해야 합니다
- 팩트 기반의 신뢰도 높은 답변을 제공할 것
- 유저의 질문을 구분하여 유연하게 검색 시스템을 구축하는것을 권장합니다
- 사용 모델은 바꿔도 되나, 비용이 더 저렴한 모델로만 변경 가능합니다

In [13]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap
from langfuse.langchain import CallbackHandler
from grading_utils import grade_predictions
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
import re
def extract_scores(llm_output: str):
    """LLM의 채점 결과에서 항목별 점수를 추출하고 총점을 계산합니다."""
    score_pattern = re.compile(r"\d+번:\s*([01](?:\.5)?)점")
    scores = score_pattern.findall(llm_output)  # ['1', '0.5', '0', '1']
    scores = list(map(float, scores))
    total = sum(scores)
    return total, scores

In [15]:
# 1. 두 개의 PDF 문서 로딩
sk_loader = PyMuPDFLoader('data/2025_SK하이닉스_지속가능경영보고서.pdf')
sk_docs = sk_loader.load()

# 2. 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=40)
sk_split_docs = text_splitter.split_documents(sk_docs)

# 3. 벡터 저장소 생성 (두 문서를 합쳐서 인덱싱)
all_docs = sk_split_docs 
embedding_model = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(all_docs, embedding_model)

In [16]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [18]:
# 4. Retriever 설정
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# 5. LLM 및 체인 구성
prompt = ChatPromptTemplate.from_template("""
다음은 문서에서 검색된 정보입니다:
{context}

이 정보를 바탕으로 사용자 질문에 답하되, 불필요한 정보는 포함하지 말고 정답 단어만 말하세요.
참조된 문서에 정보가 없고, 답변할 수 없는 경우 "없는 정보"라고 답하세요.
예시 질문 : 한국의 수도는 어디입니까?
답변 : 서울
질문: {question}
""")
parser = StrOutputParser()

chain = (
    RunnableMap({
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x["question"]
    })
    | prompt
    | llm
    | parser
)

# 6. questions.txt & qa_dataset.json 로드
import json

with open("questions.txt", "r", encoding="utf-8") as f:
    questions = [line.strip() for line in f if line.strip()]

with open("qa_dataset.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)
answers = [pair["answer"] for pair in dataset["qa_pairs"]]

print(f"질문 {len(questions)}개, 정답 {len(answers)}개 로드")

# 7. 답변 생성

predicted_answers = []
for i, q in enumerate(questions, start=1):
    answer = chain.invoke({"question": q})
    predicted_answers.append(answer)
    print(f"\n💬 질문 {i}: {q}")
    print(f"→ 예측: {answer}")

# 8. 자동 채점
n = min(len(questions), len(answers), len(predicted_answers))
print("\n✅ 자동 채점 결과:")
grading_result = grade_predictions(questions[:n], predicted_answers[:n], answers[:n])
print(grading_result)

total_score, per_item_scores = extract_scores(grading_result)
print(f"\n총점: {total_score}점 / {n}점 만점")

질문 30개, 정답 30개 로드

💬 질문 1: 보고서 발행일은 언제인가요?
→ 예측: 2025년 6월 18일

💬 질문 2: 보고 기간은 언제부터 언제까지인가요?
→ 예측: 2024년 1월 1일부터 2024년 12월 31일까지

💬 질문 3: 이해관계자 구분은 몇 개 그룹인가요?
→ 예측: 7개 그룹

💬 질문 4: 고객 소통 채널 중 하나는 무엇인가요?
→ 예측: 홈페이지

💬 질문 5: 장애인/저소득층 일자리 창출 개수는?
→ 예측: 1000개

💬 질문 6: Global ICT 인재 육성 프로그램 참여 누적 인원은?
→ 예측: 10만 명


KeyboardInterrupt: 